# <span style="color: #1F1DB5;">SPRINT 16 – Series Temporales

# <span style="color: #1F1DB5;">Notebook 1 – Series de Tiempo: Descomposición de Señal — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Comprender la estructura de una serie de tiempo: tendencia, estacionalidad y residuos.
- Aplicar descomposición de señal (aditiva y multiplicativa) con seasonaldecompose.
- Detectar estacionariedad con el test Augmented Dickey-Fuller (ADF).
- Calcular e interpretar ACF y PACF para selección de modelos.
- Aplicar promedios móviles como técnica de suavizado y baseline de predicción.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender la **estructura de una serie de tiempo**: tendencia, estacionalidad y residuos.
- Aplicar **descomposición de señal** (aditiva y multiplicativa) con `seasonal_decompose`.
- Detectar **estacionariedad** con el test Augmented Dickey-Fuller (ADF).
- Calcular e interpretar **ACF y PACF** para selección de modelos.
- Aplicar **promedios móviles** como técnica de suavizado y baseline de predicción.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Estructura de una serie de tiempo | 20 min |
Descomposición de señal | 20 min |
Estacionariedad y test ADF | 20 min |
ACF y PACF | 15 min |
Breakout Rooms | 20 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo podemos predecir el futuro usando patrones del pasado?

Piensa en las ventas de helados: suben en verano y bajan en invierno, año tras año. Eso es **estacionalidad**. Pero además, cada año se venden un poco más que el anterior: eso es la **tendencia**. Si podemos separar estos componentes, podemos entender qué está pasando realmente con nuestros datos y hacer predicciones más inteligentes.

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Estructura de una Serie de Tiempo (20 mins)

Una **serie de tiempo** es una secuencia de observaciones ordenadas cronológicamente. A diferencia de los datos tabulares tradicionales, aquí el **orden importa**: no puedes mezclar las filas aleatoriamente sin perder información.

Toda serie de tiempo se compone de:

1. **Tendencia (Trend):** La dirección general a largo plazo. ¿Los datos suben, bajan o se mantienen estables?
2. **Estacionalidad (Seasonality):** Patrones que se repiten en intervalos fijos (diarios, semanales, mensuales, anuales).
3. **Residuos (Residuals):** Lo que queda después de extraer tendencia y estacionalidad. Idealmente es ruido aleatorio.

**Analogía:** Imagina el precio de una acción. La tendencia es si la empresa va creciendo a largo plazo. La estacionalidad podrían ser las ventas navideñas que suben cada diciembre. Y los residuos son las fluctuaciones diarias impredecibles.

**Requisitos de una serie de tiempo en Python:**
- El índice debe ser de tipo `DatetimeIndex`.
- La frecuencia debe ser regular (si faltan datos, hay que interpolarlos o rellenarlos).
- Los valores deben ser numéricos.

Vamos a crear una serie de tiempo sintética que simula el número de pasajeros aéreos mensuales (inspirado en el clásico dataset AirPassengers). Esto nos permitirá ver claramente los tres componentes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Creamos una serie tipo AirPassengers (144 meses = 12 años)
np.random.seed(42)
n_months = 144
dates = pd.date_range(start='1949-01-01', periods=n_months, freq='MS')

# Componentes
trend = np.linspace(100, 500, n_months)  # tendencia creciente
seasonal = 40 * np.sin(2 * np.pi * np.arange(n_months) / 12)  # ciclo anual
noise = np.random.normal(0, 15, n_months)  # ruido

# Serie multiplicativa (más realista para pasajeros)
passengers = trend * (1 + seasonal / 200) + noise

ts = pd.Series(passengers, index=dates, name='Pasajeros')
print(f"Tipo de índice: {type(ts.index)}")
print(f"Frecuencia: {ts.index.freq}")
print(f"Periodo: {ts.index[0].date()} a {ts.index[-1].date()}")
print(f"\nPrimeros valores:\n{ts.head(12)}")

# Visualización
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts, color='steelblue')
ax.set_title('Pasajeros Aéreos Mensuales (serie sintética)', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Número de pasajeros')
plt.tight_layout()
plt.show()

Preguntas:

- ¿Puedes identificar visualmente la tendencia y la estacionalidad en la gráfica?

- ¿Por qué el índice debe ser de tipo DatetimeIndex y no simplemente números enteros?

- ¿Qué pasaría si tuviéramos meses faltantes en nuestra serie?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Descomposición de Señal (20 mins)

La **descomposición** consiste en separar una serie en sus componentes individuales. Esto nos permite:
- Entender qué parte del cambio se debe a la tendencia vs. estacionalidad.
- Detectar anomalías en los residuos.
- Preparar los datos para modelado (muchos modelos requieren series estacionarias).

Existen dos modelos de descomposición:

**Modelo Aditivo:** `Y(t) = Tendencia + Estacionalidad + Residuos`
- Cuando la amplitud estacional es **constante** a lo largo del tiempo.

**Modelo Multiplicativo:** `Y(t) = Tendencia × Estacionalidad × Residuos`
- Cuando la amplitud estacional **crece** con la tendencia (más común en la realidad).

**¿Cómo elegir?** Si la variación estacional aumenta conforme sube la serie → multiplicativo. Si se mantiene igual → aditivo.

Usemos `seasonal_decompose` de statsmodels para descomponer nuestra serie. Probaremos ambos modelos y compararemos los residuos.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Descomposición ADITIVA
decomp_add = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
decomp_add.observed.plot(ax=axes[0], title='Serie Original')
decomp_add.trend.plot(ax=axes[1], title='Tendencia')
decomp_add.seasonal.plot(ax=axes[2], title='Estacionalidad')
decomp_add.resid.plot(ax=axes[3], title='Residuos')
for ax in axes:
    ax.set_xlabel('')
fig.suptitle('Descomposición ADITIVA', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Descomposición MULTIPLICATIVA
decomp_mult = seasonal_decompose(ts, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
decomp_mult.observed.plot(ax=axes[0], title='Serie Original')
decomp_mult.trend.plot(ax=axes[1], title='Tendencia')
decomp_mult.seasonal.plot(ax=axes[2], title='Estacionalidad')
decomp_mult.resid.plot(ax=axes[3], title='Residuos')
for ax in axes:
    ax.set_xlabel('')
fig.suptitle('Descomposición MULTIPLICATIVA', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Varianza de residuos (aditivo):", decomp_add.resid.dropna().var().round(2))
print("Varianza de residuos (multiplicativo):", decomp_mult.resid.dropna().var().round(6))
print("\n→ El modelo con MENOR varianza en residuos es mejor ajuste.")

Preguntas:

- ¿Cuál de los dos modelos (aditivo o multiplicativo) ajusta mejor a nuestros datos y por qué?

- ¿Qué información nos dan los residuos sobre la calidad de la descomposición?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Estacionariedad y Test ADF (20 mins)

Una serie es **estacionaria** cuando sus propiedades estadísticas (media, varianza) no cambian con el tiempo. Esto es crucial porque:

- La mayoría de los modelos de series de tiempo (ARIMA, etc.) **requieren** estacionariedad.
- Una serie no estacionaria puede producir correlaciones espurias.
- Nos indica si necesitamos **diferenciar** la serie antes de modelar.

**Señales de NO estacionariedad:**
- Tendencia visible (media cambia).
- Varianza que crece o decrece.
- Estacionalidad fuerte.

**Test Augmented Dickey-Fuller (ADF):**
- H₀: La serie **NO es estacionaria** (tiene raíz unitaria).
- H₁: La serie **ES estacionaria**.
- Si p-value < 0.05 → rechazamos H₀ → la serie es estacionaria.

**Diferenciación:** Restar cada valor con el anterior: `diff = y(t) - y(t-1)`. Si una diferenciación no basta, se aplica dos veces (raro).

Apliquemos el test ADF a nuestra serie original y luego a su versión diferenciada para ver cómo la diferenciación transforma una serie no estacionaria en estacionaria.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def test_adf(series, nombre=''):
    """Aplica test ADF e imprime resultados interpretados."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"--- Test ADF: {nombre} ---")
    print(f"  Estadístico ADF: {result[0]:.4f}")
    print(f"  p-value: {result[1]:.6f}")
    print(f"  Lags usados: {result[2]}")
    print(f"  Observaciones: {result[3]}")
    if result[1] < 0.05:
        print(f"  ✅ ESTACIONARIA (p < 0.05, rechazamos H₀)")
    else:
        print(f"  ❌ NO ESTACIONARIA (p >= 0.05, no podemos rechazar H₀)")
    print()

# Serie original
test_adf(ts, 'Serie Original')

# Primera diferenciación
ts_diff = ts.diff().dropna()
test_adf(ts_diff, 'Primera Diferenciación')

# Visualización comparativa
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(ts, color='steelblue')
axes[0].set_title('Serie Original (no estacionaria)')
axes[0].axhline(ts.mean(), color='red', linestyle='--', label='Media')
axes[0].legend()

axes[1].plot(ts_diff, color='darkorange')
axes[1].set_title('Serie Diferenciada (estacionaria)')
axes[1].axhline(ts_diff.mean(), color='red', linestyle='--', label='Media')
axes[1].legend()
plt.tight_layout()
plt.show()

Preguntas:

- ¿Por qué es importante que una serie sea estacionaria antes de aplicar modelos como ARIMA?

- ¿Qué representa visualmente la diferencia entre una serie estacionaria y una que no lo es?

- ¿Cuántas diferenciaciones necesitó nuestra serie para volverse estacionaria?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">ACF y PACF (15 mins)

Las funciones de **autocorrelación** son herramientas fundamentales para entender la estructura temporal de una serie y elegir los parámetros de modelos ARIMA.

**ACF (Autocorrelation Function):**
- Mide la correlación entre la serie y versiones retrasadas (lags) de sí misma.
- Incluye correlaciones directas E indirectas.
- Nos ayuda a identificar el parámetro **q** (orden MA) del modelo ARIMA.

**PACF (Partial Autocorrelation Function):**
- Mide la correlación entre la serie y un lag específico, **eliminando** el efecto de los lags intermedios.
- Nos ayuda a identificar el parámetro **p** (orden AR) del modelo ARIMA.

**Reglas prácticas:**
- Si ACF decae gradualmente y PACF se corta en lag p → modelo AR(p).
- Si PACF decae gradualmente y ACF se corta en lag q → modelo MA(q).
- Si ambos decaen gradualmente → modelo ARMA(p,q).

**Promedios Móviles (Moving Averages):** Suavizan la serie calculando la media de una ventana deslizante. Útiles como:
- Técnica de visualización para ver tendencia.
- Baseline naive de predicción.

Calculemos ACF y PACF de la serie diferenciada (estacionaria) y apliquemos promedios móviles a la serie original para suavizar el ruido.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ACF y PACF de la serie diferenciada
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts_diff, lags=36, ax=axes[0], title='ACF – Serie Diferenciada')
plot_pacf(ts_diff, lags=36, ax=axes[1], title='PACF – Serie Diferenciada')
plt.tight_layout()
plt.show()

print("Interpretación:")
print("- ACF: busca dónde se corta abruptamente → sugiere q para MA")
print("- PACF: busca dónde se corta abruptamente → sugiere p para AR")
print("- Picos en lags múltiplos de 12 = estacionalidad anual")

In [ ]:
# Promedios móviles como suavizado
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ts, alpha=0.4, label='Original', color='gray')
ax.plot(ts.rolling(window=6).mean(), label='Media Móvil 6 meses', color='blue', linewidth=2)
ax.plot(ts.rolling(window=12).mean(), label='Media Móvil 12 meses', color='red', linewidth=2)
ax.set_title('Promedios Móviles como Suavizado')
ax.legend()
ax.set_xlabel('Fecha')
ax.set_ylabel('Pasajeros')
plt.tight_layout()
plt.show()

print("La media móvil de 12 meses elimina la estacionalidad anual")
print("y nos muestra solo la tendencia subyacente.")

## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (20 mins)

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

**Instrucciones:**

1. Generen una serie de tiempo con los datos proporcionados abajo (ventas mensuales de una tienda).
2. Apliquen descomposición aditiva Y multiplicativa. ¿Cuál ajusta mejor?
3. Realicen el test ADF a la serie original y a la diferenciada.
4. Grafiquen ACF y PACF de la serie estacionaria.
5. ¿Qué valores de p y q sugieren las gráficas?

In [ ]:
# Datos para el ejercicio del Breakout
np.random.seed(123)
n = 96  # 8 años mensuales
fechas = pd.date_range('2015-01-01', periods=n, freq='MS')
tendencia = np.linspace(200, 800, n)
estacional = 100 * np.sin(2 * np.pi * np.arange(n) / 12)
ruido = np.random.normal(0, 30, n)

ventas = tendencia + estacional * (tendencia / 400) + ruido
serie_ventas = pd.Series(ventas, index=fechas, name='Ventas')
serie_ventas.plot(figsize=(10, 4), title='Ventas Mensuales - Tu Serie para Analizar')
plt.show()

# --- TU CÓDIGO AQUÍ ---
# 1. Descomposición
# decomp = seasonal_decompose(serie_ventas, model='...', period=12)
# decomp.plot()

# 2. Test ADF
# test_adf(serie_ventas, 'Ventas Original')
# test_adf(serie_ventas.diff().dropna(), 'Ventas Diferenciada')

# 3. ACF / PACF
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# plot_acf(serie_ventas.diff().dropna(), lags=24, ax=axes[0])
# plot_pacf(serie_ventas.diff().dropna(), lags=24, ax=axes[1])
# plt.show()

## <span style="color: #2826DE;">Tips y buenas prácticas

- Siempre **visualiza** tu serie antes de modelar. Los ojos detectan patrones que las estadísticas no.
- Verifica la **frecuencia** de tus datos. Si hay huecos, rellena con interpolación o forward fill.
- La descomposición **multiplicativa** es más apropiada cuando la variación crece con el nivel de la serie.
- El test ADF tiene baja potencia con series cortas. Complementa con KPSS si tienes dudas.
- ACF y PACF se calculan sobre la serie **estacionaria** (diferenciada), no sobre la original.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Cuáles son los tres componentes de una serie de tiempo?

- Media, varianza y covarianza
- Tendencia, estacionalidad y residuos 
- Promedio, mediana y moda
- Señal, ruido y frecuencia


2️⃣ ¿Qué modelo de descomposición usamos cuando la amplitud estacional crece con la tendencia?

- Aditivo
- Logarítmico
- Multiplicativo 
- Exponencial


3️⃣ ¿Qué indica un p-value < 0.05 en el test ADF?

- La serie tiene tendencia
- La serie NO es estacionaria
- La serie ES estacionaria 
- El modelo es significativo


4️⃣ ¿Qué función nos ayuda a identificar el parámetro p de un modelo AR?

- ACF
- PACF 
- ADF
- Seasonal decompose


5️⃣ ¿Cuál es el propósito de la diferenciación en series de tiempo?

- Eliminar outliers
- Hacer la serie estacionaria 
- Suavizar la serie
- Predecir el futuro


6️⃣ ¿Qué ventana de media móvil elimina la estacionalidad anual en datos mensuales?

- 6 meses
- 12 meses 
- 24 meses
- 3 meses